# Exploring the Relationship Between Solar Load Forecast and Meterology Features

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"
merged_out = processed_dir / "03_merged_data.csv"

In [3]:
df = pd.read_csv(merged_out, low_memory=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (509364, 12)
Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']


,time_stamp,zone_name,actual_mw,forecast_mw,capacity_mw,time,temperature_2m,surface_pressure,cloud_cover,windspeed_10m,shortwave_radiation,year
0,2020-11-17 05:00:00+00:00,CAPITL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
1,2020-11-17 05:00:00+00:00,CENTRL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
2,2020-11-17 05:00:00+00:00,DUNWOD,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
3,2020-11-17 05:00:00+00:00,GENESE,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
4,2020-11-17 05:00:00+00:00,HUD VL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020


In [4]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()

if "time" in df.columns:
    same_time_mask = (df["time"] == df["time_stamp"]) | (df["time"].isna() & df["time_stamp"].isna())
    time_columns_identical = bool(same_time_mask.all())
    print("Time Column Like time_stamp:", time_columns_identical)

    if time_columns_identical:
        df = df.drop(columns=["time"])

print("Columns after the Check:", df.columns.tolist())
print(df.dtypes)

Time Column Like time_stamp: True
Columns after the Check: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']
time_stamp             datetime64[us, UTC]
zone_name                              str
actual_mw                          float64
forecast_mw                        float64
capacity_mw                        float64
temperature_2m                     float64
surface_pressure                   float64
cloud_cover                          int64
windspeed_10m                      float64
shortwave_radiation                float64
year                                 int64
dtype: object


In [5]:
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")
df["date_local"] = df["time_local"].dt.date
df["year_from_ts"] = df["time_local"].dt.year
df["quarter_local"] = df["time_local"].dt.quarter
df["month_local"] = df["time_local"].dt.month
df["day_local"] = df["time_local"].dt.day
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour
df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

In [6]:
df_system = df[df["zone_name"] == "SYSTEM"].copy()
df_zones = df[df["zone_name"] != "SYSTEM"].copy()
df_nyc = df[df["zone_name"] == "N.Y.C."].copy()

print("Full Shape:", df.shape)
print("System Only:", df_system.shape)
print("Zone Only:", df_zones.shape)
print("NYC Only:", df_nyc.shape)

Full Shape: (509364, 21)
System Only: (42447, 21)
Zone Only: (466917, 21)
NYC Only: (42447, 21)


In [7]:
schema_check = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [(df[c].isna().mean() * 100).round(6) for c in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

duplicate_summary = pd.DataFrame({
    "check": [
        "duplicate_full_rows_all",
        "duplicate_time_zone_all",
        "duplicate_timestamps_all",
        "duplicate_time_zone_zones_only",
        "duplicate_timestamps_system_only",
    ],
    "count": [
        df.duplicated().sum(),
        df.duplicated(subset=["time_stamp", "zone_name"]).sum(),
        df.duplicated(subset=["time_stamp"]).sum(),
        df_zones.duplicated(subset=["time_stamp", "zone_name"]).sum(),
        df_system.duplicated(subset=["time_stamp"]).sum(),
    ]
})

ts_min = df["time_stamp"].min()
ts_max = df["time_stamp"].max()
full_hours = pd.date_range(start=ts_min, end=ts_max, freq="h", tz="UTC")
observed_hours = pd.Index(df["time_stamp"].dropna().sort_values().unique())
missing_hours = full_hours.difference(observed_hours)

coverage_summary = pd.DataFrame({
    "expected_hour_count": [len(full_hours)],
    "observed_hour_count": [len(observed_hours)],
    "missing_hour_count": [len(missing_hours)],
})

missing_hours_df = pd.DataFrame({"missing_time_stamp": missing_hours})
missing_hours_df["time_local"] = missing_hours_df["missing_time_stamp"].dt.tz_convert("America/New_York")

schema_check, duplicate_summary, coverage_summary, missing_hours_df.head(20)

(                 column                             dtype  missing_count  \
 4           capacity_mw                           float64         498648   
 2             actual_mw                           float64           8544   
 3           forecast_mw                           float64           3456   
 7           cloud_cover                             int64              0   
 12           date_local                            object              0   
 16            day_local                             int32              0   
 17      dayofweek_local                             int32              0   
 18           hour_local                             int32              0   
 20    is_daylight_proxy                             int64              0   
 19           is_weekend                             int64              0   
 15          month_local                             int32              0   
 14        quarter_local                             int32              0   

In [15]:
zone_coverage = (
    df.groupby("zone_name", as_index=False)
      .agg(
          rows=("time_stamp", "size"),
          unique_hours=("time_stamp", "nunique"),
          actual_missing=("actual_mw", lambda s: s.isna().sum()),
          forecast_missing=("forecast_mw", lambda s: s.isna().sum()),
          capacity_missing=("capacity_mw", lambda s: s.isna().sum()),
          actual_missing_pct=("actual_mw", lambda s: round(s.isna().mean() * 100, 4)),
          forecast_missing_pct=("forecast_mw", lambda s: round(s.isna().mean() * 100, 4)),
          capacity_missing_pct=("capacity_mw", lambda s: round(s.isna().mean() * 100, 4)),
      )
      .sort_values("zone_name")
)

zone_coverage

,zone_name,rows,unique_hours,actual_missing,forecast_missing,capacity_missing,actual_missing_pct,forecast_missing_pct,capacity_missing_pct
0,CAPITL,42447,42447,712,288,41554,1.6774,0.6785,97.8962
1,CENTRL,42447,42447,712,288,41554,1.6774,0.6785,97.8962
2,DUNWOD,42447,42447,712,288,41554,1.6774,0.6785,97.8962
3,GENESE,42447,42447,712,288,41554,1.6774,0.6785,97.8962
4,HUD VL,42447,42447,712,288,41554,1.6774,0.6785,97.8962
5,LONGIL,42447,42447,712,288,41554,1.6774,0.6785,97.8962
6,MHK VL,42447,42447,712,288,41554,1.6774,0.6785,97.8962
7,MILLWD,42447,42447,712,288,41554,1.6774,0.6785,97.8962
8,N.Y.C.,42447,42447,712,288,41554,1.6774,0.6785,97.8962
9,NORTH,42447,42447,712,288,41554,1.6774,0.6785,97.8962


In [9]:
for frame in [df, df_system, df_zones, df_nyc]:
    frame["forecast_error_mw"] = frame["actual_mw"] - frame["forecast_mw"]
    frame["absolute_error_mw"] = (frame["actual_mw"] - frame["forecast_mw"]).abs()

    frame["ape"] = np.where(
        frame["actual_mw"].abs() > 0,
        (frame["absolute_error_mw"] / frame["actual_mw"].abs()) * 100,
        np.nan
    )

    frame["smape"] = np.where(
        (frame["actual_mw"].abs() + frame["forecast_mw"].abs()) > 0,
        200 * frame["absolute_error_mw"] / (frame["actual_mw"].abs() + frame["forecast_mw"].abs()),
        np.nan
    )

def error_summary(frame, scope_name):
    return pd.DataFrame({
        "scope": [scope_name],
        "rows": [len(frame)],
        "nonmissing_error_rows": [frame["forecast_error_mw"].notna().sum()],
        "me": [frame["forecast_error_mw"].mean()],
        "mae": [frame["absolute_error_mw"].mean()],
        "rmse": [np.sqrt(np.nanmean(np.square(frame["forecast_error_mw"])))],
        "median_ae": [frame["absolute_error_mw"].median()],
        "smape_mean": [frame["smape"].mean()],
        "ape_mean": [frame["ape"].mean()],
    })

error_summary_table = pd.concat([
    error_summary(df, "all_rows"),
    error_summary(df_zones, "zones_only"),
    error_summary(df_system, "system_only"),
    error_summary(df_nyc, "nyc_only"),
], ignore_index=True)

error_summary_table

,scope,rows,nonmissing_error_rows,me,mae,rmse,median_ae,smape_mean,ape_mean
0,all_rows,509364,497364,-0.363698,16.783057,54.424400,0.38,39.614424,91.414071
1,zones_only,466917,455917,-0.198992,10.878079,26.863974,0.32,40.306171,86.090642
2,system_only,42447,41447,-2.175471,81.737819,166.149873,6.99,32.200133,148.127554
3,nyc_only,42447,41447,-1.167278,10.454884,21.995315,0.67,38.979720,79.451326


In [10]:
zone_summary = (
    df_zones.groupby("zone_name", as_index=False)
      .agg(
          rows=("time_stamp", "size"),
          unique_hours=("time_stamp", "nunique"),
          actual_mean=("actual_mw", "mean"),
          actual_median=("actual_mw", "median"),
          actual_max=("actual_mw", "max"),
          forecast_mean=("forecast_mw", "mean"),
          forecast_median=("forecast_mw", "median"),
          forecast_max=("forecast_mw", "max"),
          mae=("absolute_error_mw", "mean"),
          rmse=("forecast_error_mw", lambda s: np.sqrt(np.nanmean(np.square(s)))),
          me=("forecast_error_mw", "mean"),
          smape_mean=("smape", "mean"),
          daylight_actual_mean=("actual_mw", lambda s: s[df_zones.loc[s.index, "is_daylight_proxy"] == 1].mean()),
          nighttime_actual_mean=("actual_mw", lambda s: s[df_zones.loc[s.index, "is_daylight_proxy"] == 0].mean()),
      )
      .sort_values("actual_mean", ascending=False)
)

zone_summary

,zone_name,rows,unique_hours,actual_mean,actual_median,actual_max,forecast_mean,forecast_median,forecast_max,mae,rmse,me,smape_mean,daylight_actual_mean,nighttime_actual_mean
5,LONGIL,42447,42447,114.282697,3.00,821.67,115.812237,6.30,841.17,23.193983,47.853459,-1.938492,40.942147,210.714002,2.025356
1,CENTRL,42447,42447,79.089408,1.81,854.18,78.174527,2.45,867.10,17.071595,37.757230,0.650796,42.290204,146.356513,0.782614
0,CAPITL,42447,42447,76.971081,2.16,646.75,76.530934,3.77,643.89,16.333617,33.701364,0.190462,42.009168,142.151084,1.093919
4,HUD VL,42447,42447,76.720386,2.21,662.28,78.067970,3.91,664.46,14.670732,30.964964,-1.647047,38.708236,141.514273,1.292707
6,MHK VL,42447,42447,61.791764,1.83,605.78,59.827285,2.27,612.09,12.343684,25.655760,1.806083,39.427570,114.177197,0.808981
8,N.Y.C.,42447,42447,53.713888,1.50,423.74,54.669531,2.77,441.56,10.454884,21.995315,-1.167278,38.979720,99.203790,0.758311
3,GENESE,42447,42447,48.838276,0.96,578.23,48.975447,1.53,565.59,10.420241,23.402069,-0.292854,42.590873,90.448907,0.398624
10,WEST,42447,42447,41.994934,0.97,350.71,41.357655,1.37,374.44,9.086960,19.400025,0.522287,42.462712,77.840314,0.266711
2,DUNWOD,42447,42447,13.644407,0.43,116.99,13.747213,0.73,120.09,2.731371,5.910062,-0.146134,39.366078,25.146082,0.255110
7,MILLWD,42447,42447,9.810924,0.31,94.33,9.960694,0.51,95.77,1.887804,4.066002,-0.184925,38.472361,18.088792,0.174515
